# Your own shape: from a drawing to a cracked part

This tutorial takes a part you design yourself through every stage of a
quasi-static phase-field fracture calculation:

1. **geometry**: describe the shape in a Gmsh `.geo` file;
2. **mesh**: cut it into triangles and name its edges;
3. **material definition**;
4. **material applied to the mesh** through named regions;
5. **boundary conditions and loads**;
6. **run the simulation and save the results**;
7. **view the data**; and
8. **post-process**: force against displacement, the crack and the deformed shape.

The part is a plate 20 mm wide and 16 mm high with a 4 mm notch in its left
edge and a hole of radius 2 mm above and to the right of the notch tip. The
bottom edge is clamped and the top edge is pulled upwards. A crack starts at
the notch, and the calculation shows how the hole changes its path.

**Units.** PhAST has no built-in units; choose a consistent set. Here lengths
are in mm and forces in N, so stiffness is in MPa (N/mm²) and toughness in
N/mm. The model is two-dimensional (plane strain): forces are per millimetre
of out-of-plane thickness. The solve takes about two minutes on a laptop CPU.

## 0. Install PhAST by the portable source route

Use Python >=3.10 with compatible dependency wheels. For a new source checkout:

~~~bash
git clone https://github.com/CEMS-Lab/PhAST.git
cd PhAST
python3 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
python -m pip install -e . jupyterlab
python tools/run_sanitizer.py
python -m phast doctor
jupyter lab docs/tutorial/notebook_own_shape.ipynb
~~~

On Windows PowerShell, activate with <code>.venv\Scripts\Activate.ps1</code>.
Conda and Docker routes are in the [installation guide](../install.md). The base
CPU installation is sufficient; Gmsh is needed to mesh the shape.

For an existing checkout or a documentation build, use that checkout and its
prepared environment. Do not clone or install an older release over it. The
clone command above selects public `main`, which is mutable; local changes
are not assumed to be published there.

In [ ]:
from __future__ import annotations
import os
from pathlib import Path
import shutil
import subprocess
import sys
import time

import matplotlib.pyplot as plt
import matplotlib.tri as mtri
import meshio
import numpy as np


def find_repository_root(start: Path) -> Path:
    # Locate a source checkout from the notebook working directory.
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "examples").is_dir():
            return candidate
    raise RuntimeError(
        "Run this notebook from a cloned PhAST repository. "
        "See the installation commands above."
    )

ROOT = find_repository_root(Path.cwd())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
os.environ["PYTHONPATH"] = str(ROOT / "src") + os.pathsep + os.environ.get("PYTHONPATH", "")

import phast
import io
from IPython.display import Image, display

plt.rcParams.update({"font.family": "STIXGeneral", "mathtext.fontset": "stix", "font.size": 10})

def show(fig):
    # Render to PNG so figures appear whichever matplotlib backend is active.
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", dpi=110, bbox_inches="tight")
    plt.close(fig)
    display(Image(buffer.getvalue()))

def require_finite(values, name):
    # Reject empty or non-finite data rather than disguising a failed solve.
    array = np.asarray(values)
    if array.size == 0 or not np.isfinite(array).all():
        raise ValueError(
            f"Empty or non-finite values in {name}; inspect the solver log "
            "before plotting or interpreting this run."
        )
    return array

output_root = Path(os.environ.get("PHAST_NOTEBOOK_OUTPUT_ROOT", ROOT / "runs"))
base = output_root.expanduser().resolve() / "tutorial_own_shape"
RUN_DIR = base
suffix = 2
while RUN_DIR.exists():
    RUN_DIR = Path(f"{base}_{suffix}")
    suffix += 1
RUN_DIR.mkdir(parents=True)

GEO_PATH = RUN_DIR / "mesh.geo"
MESH_PATH = RUN_DIR / "mesh.msh"
CONFIG_PATH = RUN_DIR / "setup.yaml"
SOLVE_DIR = RUN_DIR / "run"

print(f"Repository: {ROOT.name}")
print(f"Tutorial output: {RUN_DIR}")
print(f"PhAST import: {Path(phast.__file__).resolve().relative_to(ROOT)}")

## 1. Geometry: describe the shape

A Gmsh `.geo` file lists **points**, joins them with **lines** and **arcs**,
closes the lines into **loops**, and fills a loop to make a **surface**. A
second loop inside the first becomes a hole. Two parts matter most for PhAST:

* **Element size.** A crack in PhAST is a smooth band about `l0` wide (see
  section 3), so the elements where the crack may run must be smaller than
  about `l0 / 2`. A *box field* makes the elements fine in a band and coarse
  elsewhere, which keeps the run fast.
* **Physical names.** `Physical Curve("bottom")` gives the bottom edge a name.
  PhAST does not use node numbers: boundary conditions and materials refer to
  these names.

Edit the numbers in this cell to change the shape, then run the notebook again.

In [ ]:
GEO_TEXT = '''
// Plate 20 mm x 16 mm, 4 mm edge notch on the left, hole of radius 2 mm.
h_far  = 0.8;   // element size away from the crack path
h_fine = 0.12;  // element size along the expected crack path (about l0 / 2.5)

Point(1) = { 0.0,  0.0, 0, h_far};
Point(2) = {20.0,  0.0, 0, h_far};
Point(3) = {20.0, 16.0, 0, h_far};
Point(4) = { 0.0, 16.0, 0, h_far};
Point(5) = { 0.0,  8.1, 0, h_fine};
Point(6) = { 4.0,  8.0, 0, h_fine};   // notch tip
Point(7) = { 0.0,  7.9, 0, h_fine};
Line(1) = {1, 2};  Line(2) = {2, 3};  Line(3) = {3, 4};
Line(4) = {4, 5};  Line(5) = {5, 6};  Line(6) = {6, 7};  Line(7) = {7, 1};
Curve Loop(1) = {1, 2, 3, 4, 5, 6, 7};

Point(10) = {12.0, 10.5, 0, h_fine};  // hole centre
Point(11) = {14.0, 10.5, 0, h_fine};
Point(12) = {10.0, 10.5, 0, h_fine};
Circle(8) = {11, 10, 12};  Circle(9) = {12, 10, 11};
Curve Loop(2) = {8, 9};

Plane Surface(1) = {1, 2};

// Fine elements in a band from the notch tip, past the hole, to the right edge
Field[1] = Box;
Field[1].VIn = h_fine;  Field[1].VOut = h_far;
Field[1].XMin = 3.0;  Field[1].XMax = 20.0;
Field[1].YMin = 6.5;  Field[1].YMax = 12.5;
Field[1].Thickness = 2.0;
Background Field = 1;
Mesh.CharacteristicLengthExtendFromBoundary = 0;

// Names PhAST uses
Physical Curve("bottom") = {1};
Physical Curve("top")    = {3};
Physical Surface("body") = {1};
'''
GEO_PATH.write_text(GEO_TEXT.lstrip(), encoding="utf-8")
print(f"Wrote {GEO_PATH.name} ({len(GEO_TEXT.splitlines())} lines)")

## 2. Mesh: cut the shape into triangles

`gmsh -2` makes a two-dimensional mesh of first-order triangles, the element
PhAST supports in full. Then ask PhAST what it sees in the file:
`phast.inspect_mesh` lists the named groups, two edges (dimension 1) and one
surface (dimension 2). If a name you expect is missing, fix it in the `.geo`
text first; this is the most common mistake with a new shape.

In [ ]:
gmsh_executable = shutil.which("gmsh")
if gmsh_executable is None:
    raise RuntimeError(
        "Gmsh was not found. Reinstall the PhAST environment or follow "
        "the Gmsh recovery advice in docs/install.md."
    )
completed = subprocess.run(
    [gmsh_executable, str(GEO_PATH), "-2", "-format", "msh2", "-o", str(MESH_PATH)],
    cwd=ROOT, text=True, capture_output=True, timeout=120,
)
if completed.returncode != 0:
    raise RuntimeError(completed.stderr or completed.stdout)

summary = phast.inspect_mesh(str(MESH_PATH))
for name, group in summary["named_groups"].items():
    print(f"{name:7s} dimension {group['dimension']}  {group['cell_counts']}")

In [ ]:
raw = meshio.read(MESH_PATH)
xy = require_finite(raw.points[:, :2], "mesh coordinates")
tri = np.concatenate([c.data for c in raw.cells if c.type == "triangle"])
triang = mtri.Triangulation(xy[:, 0], xy[:, 1], tri)

edges = {}
for name, (tag, dim) in raw.field_data.items():
    if dim == 1:
        nodes = set()
        for block, tags in zip(raw.cells, raw.cell_data["gmsh:physical"]):
            if block.type == "line":
                nodes |= set(block.data[tags == tag].ravel().tolist())
        edges[name] = np.array(sorted(nodes))

fig, ax = plt.subplots(figsize=(6.2, 5.0), constrained_layout=True)
ax.triplot(triang, lw=0.25, color="#4d6b8a")
for (name, nodes), colour in zip(edges.items(), ["#c0392b", "#1e8449"]):
    ax.plot(xy[nodes, 0], xy[nodes, 1], "o", ms=3, color=colour, label=f'"{name}"')
ax.set_aspect("equal")
ax.set_xlabel("$x$ (mm)")
ax.set_ylabel("$y$ (mm)")
ax.set_title(f"{len(tri)} triangles, {len(xy)} nodes")
ax.legend(loc="upper right", frameon=True)
show(fig)

## 3. Material definition

| Symbol | Meaning | Value |
|---|---|---:|
| \(E\) | Young's modulus, the stiffness | 3000 MPa |
| \(\nu\) | Poisson's ratio | 0.3 |
| \(G_c\) | fracture toughness: energy to make 1 mm² of crack | 0.3 N/mm |
| \(\ell_0\) | width scale of the smeared crack band | 0.3 mm |

PhAST does not draw the crack as a line. It solves for a damage field \(d\)
between 0 (intact) and 1 (broken); the crack is the band where \(d\) is close
to 1. The band is a few \(\ell_0\) wide, so the elements on the crack path must
be smaller than about \(\ell_0/2\); here they are 0.12 mm.
`energy_split="amor"` means only stretching and shearing open cracks, and
`pf_model="AT2"` selects the standard damage model. These values describe a
glassy polymer for teaching; they are not calibrated for another material.

In [ ]:
pmma = dict(E=3000.0, nu=0.3, Gc=0.3, l0=0.3, energy_split="amor", pf_model="AT2")

edge = np.linalg.norm(xy[tri[:, 1]] - xy[tri[:, 0]], axis=1)
print(f"smallest element edge {edge.min():.3f} mm; l0 / 2 = {pmma['l0'] / 2:.3f} mm")

## 4. Apply the material to the mesh

PhAST links physics to the mesh through **regions**. A region is a name used
in the set-up, mapped to a named group in the mesh file (`from_mesh="..."`) or
to the whole domain (`kind="domain"`). The material is then assigned to a
region.

**Different materials in different regions.** This route applies one material
to the whole domain; calling `.material(...)` again for another region raises
an error. For a stiff or tough inclusion, use a schema-v2 deck with element
regions, as in [heterogeneous material fields](05_heterogeneous_material_fields.md)
and the [layered DCB notebook](notebook_layered_dcb.ipynb).

In [ ]:
problem = phast.Problem("Own shape: notched plate with a hole").mesh(str(MESH_PATH))

problem.region("body", kind="domain")          # the whole plate
problem.region("bottom", from_mesh="bottom")   # edge named "bottom" in the .geo text
problem.region("top", from_mesh="top")         # edge named "top" in the .geo text

problem.material("pmma", region="body", **pmma)
print(problem.config.material)

## 5. Boundary conditions and loads

* **Support:** `fix` the `bottom` edge in \(x\) and \(y\).
* **Load:** prescribe a `displacement` of the `top` edge in \(y\), pulling it up.

**The load value is a rate.** In the `simple` load protocol the prescribed
`value` is multiplied by time, and time grows by `dt` each increment. With
`value = 0.003`, `dt = 1` and 40 increments, the top edge ends at
0.003 × 40 = 0.12 mm. Setting `value` to the final displacement would pull the
part 40 times too far.

The analysis step sets the protocol, the number of increments and which
boundary conditions are active. The force on the top edge is recorded at every
increment, and `validate_setup()` checks names, regions and conditions without
meshing or solving. `problem.save(...)` writes the same set-up as a YAML deck.

In [ ]:
problem.boundary_condition("fix", region="bottom", dof="x", name="clamp_x")
problem.boundary_condition("fix", region="bottom", dof="y", name="clamp_y")
problem.boundary_condition("displacement", region="top", dof="y", value=0.003, name="pull")

n_increments, dt, rate = 40, 1.0, 0.003
problem.analysis_step("load", kind="quasi_static",
                      controls={"protocol": "simple", "num_steps": n_increments, "dt": dt},
                      active_boundary_conditions=["clamp_x", "clamp_y", "pull"])
problem.outputs(histories=[{"name": "reaction_force", "region": "top", "dof": "y"}])
problem.solver("quasi_static")
problem.device("cpu")

problem.validate_setup()
problem.save(CONFIG_PATH)
print(f"final top displacement = {rate} x {n_increments} x {dt} = {rate * n_increments * dt:.3f} mm")
print(f"set-up is consistent; saved as {CONFIG_PATH.name}")

## 6. Run the simulation and save the results

The solver is **quasi-static**: the load is applied slowly, so inertia is
ignored. Each increment alternates between two solves until they agree: the
displacement for the current damage, then the damage for the current
displacement. Damage can only grow. The solver prints progress lines and a
summary; the run takes about two minutes on a laptop CPU.

In [ ]:
t0 = time.time()
problem.run(output_dir=str(SOLVE_DIR))
print(f"finished in {time.time() - t0:.0f} s")
for path in sorted(SOLVE_DIR.iterdir()):
    print(f"{path.name:32s} {path.stat().st_size / 1e3:9.1f} kB")

The run folder is self-contained: `config.yaml` is the set-up as PhAST
understood it with every default filled in, `run_lockfile.json` fingerprints
the set-up and software versions, `mesh.msh` is the mesh used,
`result.vtkhdf` holds the stored fields (readable by PhAST and ParaView), and
the CSV files hold one row per increment.

## 7. View the data

`phast.load_result` opens a run folder without rerunning anything. It lists
the **histories** (one value per increment) and the **fields** (one value per
node or element at each stored increment).

In [ ]:
result = phast.load_result(SOLVE_DIR)
print("histories:", result.history_names())
print("fields:   ", result.field_names())

# result.vtkhdf numbers its stored steps 0, 1, 2, ...; this run stores every increment.
stored_steps = np.asarray(result.steps())
history = result.history_columns()
result_file = SOLVE_DIR / "result.vtkhdf"
print(f"{result_file.name}: {len(stored_steps)} stored steps; "
      f"history columns include {sorted(history)[:6]}")

rows = result.history("load_displacement")
for r in rows[:3]:
    print({k: r[k] for k in ("step", "displacement", "reaction_kN", "max_d")})

d_last = require_finite(result.field("damage", step=-1), "final damage")
u_last = require_finite(result.field("displacement", step=-1), "final displacement")
print(f"damage at the last increment: {d_last.shape[0]} nodal values, max {d_last.max():.3f}")
print(f"displacement at the last increment: shape {u_last.shape}")

## 8. Post-process the results

### Force against displacement
The force rises while the plate stretches elastically, peaks when the crack
starts to grow from the notch, and drops as the crack runs through the plate.
The stored column `reaction_kN` is the size of the reaction divided by 1000:
with forces in N it is in kN per mm of thickness. The plot multiplies it by
1000 to show N per mm.

In [ ]:
u_top = require_finite([float(r["displacement"]) for r in rows], "displacement history")
force = 1000.0 * require_finite([float(r["reaction_kN"]) for r in rows], "reaction history")
d_max = require_finite([float(r["max_d"]) for r in rows], "damage history")
i_peak = int(force.argmax())

fig, ax = plt.subplots(1, 2, figsize=(9.0, 3.4), constrained_layout=True)
ax[0].plot(u_top, force, "-o", ms=3, color="#1f5fa8")
ax[0].plot(u_top[i_peak], force[i_peak], "o", ms=8, mfc="none", color="#c0392b")
ax[0].annotate(f"peak {force[i_peak]:.0f} N/mm\nat {u_top[i_peak]:.3f} mm",
               (u_top[i_peak], force[i_peak]), xytext=(10, -30), textcoords="offset points")
ax[0].set_xlabel("top-edge displacement (mm)")
ax[0].set_ylabel("reaction force (N per mm thickness)")
ax[1].plot(u_top, d_max, "-o", ms=3, color="#b2182b")
ax[1].set_xlabel("top-edge displacement (mm)")
ax[1].set_ylabel("largest damage $d$")
ax[1].set_ylim(-0.02, 1.05)
show(fig)

### The crack
`result.field("damage", step=k)` returns the damage at every node after
increment `k`. The three pictures are before the peak, just after it, and at
the end. The crack leaves the notch tip, bends into the hole and continues
from its far side to the right edge: the hole is a weak point that attracts
the crack.

In [ ]:
steps = [max(i_peak - 3, 0), min(i_peak + 3, len(rows) - 1), len(rows) - 1]
fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.6), constrained_layout=True)
for ax, k in zip(axes, steps):
    d = require_finite(result.field("damage", step=k), f"damage at increment {k + 1}")
    im = ax.tripcolor(triang, d, shading="gouraud", cmap="Reds", vmin=0.0, vmax=1.0)
    ax.set_aspect("equal")
    ax.set_title(f"increment {k + 1}: $u$ = {u_top[k]:.3f} mm")
    ax.set_xlabel("$x$ (mm)")
axes[0].set_ylabel("$y$ (mm)")
fig.colorbar(im, ax=axes, label="damage $d$ (0 intact, 1 broken)", shrink=0.85)
show(fig)

### Deformed shape
Magnifying the stored displacement 40 times shows the plate opening along
the crack.

In [ ]:
scale = 40.0
deformed = mtri.Triangulation(xy[:, 0] + scale * u_last[:, 0], xy[:, 1] + scale * u_last[:, 1], tri)
fig, ax = plt.subplots(figsize=(5.6, 5.0), constrained_layout=True)
ax.tripcolor(deformed, d_last, shading="gouraud", cmap="Reds", vmin=0, vmax=1)
ax.set_aspect("equal")
ax.set_title(f"deformed shape, displacement x {scale:.0f}")
ax.set_xlabel("$x$ (mm)")
ax.set_ylabel("$y$ (mm)")
show(fig)

## 9. The same run from a YAML input deck

`problem.save(...)` in section 5 wrote the set-up as a schema-v2 YAML deck. A
deck is the form to share, to run on a cluster, or to repeat exactly. Check it
in a second, then run it from a terminal:

~~~bash
python -m phast run runs/tutorial_own_shape/setup.yaml --validate-only
python -m phast run runs/tutorial_own_shape/setup.yaml --device cpu --output_dir runs/own_shape_yaml
~~~

On a GPU machine, use `--device cuda`.

In [ ]:
check = subprocess.run(
    [sys.executable, "-m", "phast", "run", str(CONFIG_PATH), "--validate-only"],
    capture_output=True, text=True, cwd=ROOT,
)
print((check.stdout or check.stderr).strip().replace(str(RUN_DIR), RUN_DIR.name))

## 10. Your own shape: checklist

1. Edit the `.geo` text in section 1: points, lines, loops, surface.
2. Name every edge you will hold or load, and the surface, with `Physical` lines.
3. Make the elements on the likely crack path smaller than \(\ell_0/2\).
4. Mesh, and check the names with `phast.inspect_mesh`.
5. Set the material; a smaller \(\ell_0\) gives a sharper crack and needs finer elements.
6. Choose the load rate and the number of increments so that the final
   displacement is what you want and the peak force falls after at least 10 to
   15 increments.
7. Run, then read the folder with `phast.load_result`.

The mesh can also come from another mesher, as long as it writes Gmsh `.msh`
with named physical groups; see [mesh formats](../user_guide/meshes.md).